# Q2 - Batch Normalization და Layer Normalization

In [ ]:
!git clone https://github.com/AvtandilSh1/ML_Assignment.git
import sys
sys.path.insert(0, '/kaggle/working/ML_Assignment')
import os
os.chdir('/kaggle/working/ML_Assignment')

In [ ]:
import urllib.request, tarfile, os

cifar_dir = '/kaggle/working/ML_Assignment/cs231n/datasets/cifar-10-batches-py'
if not os.path.exists(cifar_dir):
    url = 'http://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
    tar_path = '/kaggle/working/ML_Assignment/cs231n/datasets/cifar-10-python.tar.gz'
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(url, tar_path)
    print('Extracting...')
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall('/kaggle/working/ML_Assignment/cs231n/datasets/')
    os.remove(tar_path)
    print('Done.')
else:
    print('CIFAR-10 ready.')

In [ ]:
import numpy as np
from cs231n.layers import *
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

def rel_error(x, y):
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

print('იმპორტი წარმატებულია')

## Batchnorm Forward - შემოწმება

In [ ]:
np.random.seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
X = np.random.randn(N, D1)
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)
a = np.maximum(0, X.dot(W1)).dot(W2)

gamma = np.ones((D3,))
beta = np.zeros((D3,))
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print('means (should be ~0):', a_norm.mean(axis=0))
print('stds  (should be ~1):', a_norm.std(axis=0))

## Batchnorm Backward - Gradient Check

In [ ]:
np.random.seed(231)
N, D = 4, 5
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)
bn_param = {'mode': 'train'}

fx = lambda x: batchnorm_forward(x, gamma, beta, bn_param)[0]
fg = lambda a: batchnorm_forward(x, a, beta, bn_param)[0]
fb = lambda b: batchnorm_forward(x, gamma, b, bn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma.copy(), dout)
db_num = eval_numerical_gradient_array(fb, beta.copy(), dout)

_, cache = batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = batchnorm_backward(dout, cache)

print('dx error (expect <1e-8):    ', rel_error(dx_num, dx))
print('dgamma error (expect <1e-8):', rel_error(da_num, dgamma))
print('dbeta error (expect <1e-8): ', rel_error(db_num, dbeta))

## Batchnorm Backward Alt - სიჩქარის შედარება

In [ ]:
import time
np.random.seed(231)
N, D = 100, 500
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)
bn_param = {'mode': 'train'}
out, cache = batchnorm_forward(x, gamma, beta, bn_param)

t1 = time.time()
dx1, dgamma1, dbeta1 = batchnorm_backward(dout, cache)
t2 = time.time()
dx2, dgamma2, dbeta2 = batchnorm_backward_alt(dout, cache)
t3 = time.time()

print('dx difference:     ', rel_error(dx1, dx2))
print('dgamma difference: ', rel_error(dgamma1, dgamma2))
print('dbeta difference:  ', rel_error(dbeta1, dbeta2))
print('speedup: %.2fx' % ((t2 - t1) / (t3 - t2)))

## FullyConnectedNet + Batchnorm - Gradient Check

In [ ]:
np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

for reg in [0, 3.14]:
    print('reg =', reg)
    model = FullyConnectedNet([H1, H2], input_dim=D, num_classes=C,
                              reg=reg, weight_scale=5e-2, dtype=np.float64,
                              normalization='batchnorm')
    loss, grads = model.loss(X, y)
    print('loss:', loss)
    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print(f'  {name}: {rel_error(grad_num, grads[name]):.2e}')

## Layernorm Forward - შემოწმება

In [ ]:
np.random.seed(231)
N, D1, D2, D3 = 4, 50, 60, 3
X = np.random.randn(N, D1)
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)
a = np.maximum(0, X.dot(W1)).dot(W2)

gamma = np.ones(D3)
beta = np.zeros(D3)
a_norm, _ = layernorm_forward(a, gamma, beta, {})
print('means per row (should be ~0):', a_norm.mean(axis=1))
print('stds per row  (should be ~1):', a_norm.std(axis=1))

## Layernorm Backward - Gradient Check

In [ ]:
np.random.seed(231)
N, D = 4, 5
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)
ln_param = {}

fx = lambda x: layernorm_forward(x, gamma, beta, ln_param)[0]
fg = lambda a: layernorm_forward(x, a, beta, ln_param)[0]
fb = lambda b: layernorm_forward(x, gamma, b, ln_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma.copy(), dout)
db_num = eval_numerical_gradient_array(fb, beta.copy(), dout)

_, cache = layernorm_forward(x, gamma, beta, ln_param)
dx, dgamma, dbeta = layernorm_backward(dout, cache)

print('dx error (expect <1e-8):    ', rel_error(dx_num, dx))
print('dgamma error (expect <1e-8):', rel_error(da_num, dgamma))
print('dbeta error (expect <1e-8): ', rel_error(db_num, dbeta))

## სწავლება - Batchnorm vs ჩვეულებრივი ქსელი

In [ ]:
np.random.seed(231)
data = get_CIFAR10_data()
hidden_dims = [100, 100, 100, 100, 100]
small_data = {
    'X_train': data['X_train'][:1000],
    'y_train': data['y_train'][:1000],
    'X_val':   data['X_val'],
    'y_val':   data['y_val'],
}

bn_model = FullyConnectedNet(hidden_dims, weight_scale=2e-2, normalization='batchnorm')
model    = FullyConnectedNet(hidden_dims, weight_scale=2e-2, normalization=None)

bn_solver = Solver(bn_model, small_data, num_epochs=10, batch_size=50,
                   update_rule='adam', optim_config={'learning_rate': 1e-3}, verbose=False)
bn_solver.train()

solver = Solver(model, small_data, num_epochs=10, batch_size=50,
                update_rule='adam', optim_config={'learning_rate': 1e-3}, verbose=False)
solver.train()

print('Batchnorm val acc: ', max(bn_solver.val_acc_history))
print('Baseline  val acc: ', max(solver.val_acc_history))